In [27]:
for i in range(3):
    url_df = 'https://fbref.com/en/comps/Big5/2022-2023/2022-2023-Big-5-European-Leagues-Stats'
    url_df = 'https://fbref.com/en/comps/Big5/20' + str(year - 1) + '-20' + str(year)
    temp = pd.read_html
    stats = pd.read_html(url_df)[0][['Squad', 'LgRk']]

In [28]:
stats 

,Squad,LgRk
0,Napoli,1
1,Manchester City,1
2,Barcelona,1
3,Paris S-G,1
4,Arsenal,2
...,...,...
93,Southampton,20
94,Elche,20
95,Troyes,19
96,Sampdoria,20


In [30]:
df = pd.read_csv("finalized_players.csv")
df.head(10)

,Player Name,Pos,Squad,League,90s,Tkl,TklW,Def 3rd,Mid 3rd,Att 3rd,...,Off,Crs,PKcon,OG,Recov,Won,Lost,Won%,Season,Rating
0,Max Aarons,DF,Bournemouth,Premier League,13.7,29,19,20,7,2,...,2,13,1,0,75,5,11,31.3,2023-2024,6.25
1,Yunis Abdelhamid,DF,Reims,Ligue 1,30.9,64,35,36,23,5,...,0,3,0,1,149,61,37,62.2,2023-2024,6.77
2,Salis Abdul Samed,MF,Lens,Ligue 1,16.9,21,14,8,10,3,...,0,3,3,0,89,2,7,22.2,2023-2024,6.22
3,Laurent Abergel,MF,Lorient,Ligue 1,31.8,85,52,43,34,8,...,1,34,0,0,226,15,14,51.7,2023-2024,6.91
4,Abner,DF,Betis,La Liga,15.6,25,19,15,9,1,...,2,26,1,0,79,14,10,58.3,2023-2024,6.41
5,Abdel Abqar,DF,Alavés,La Liga,25.7,35,19,28,5,2,...,1,2,2,1,109,39,36,52.0,2023-2024,6.52
6,Francesco Acerbi,DF,Inter,Serie A,26.5,22,13,16,6,0,...,3,9,0,0,102,70,37,65.4,2023-2024,6.95
7,Marcos Acuña,DF,Sevilla,La Liga,14.4,33,21,20,12,1,...,1,82,0,0,99,13,11,54.2,2023-2024,6.64
8,Tosin Adarabioyo,DF,Fulham,Premier League,18.0,21,11,16,5,0,...,0,1,0,0,43,56,28,66.7,2023-2024,6.80
9,Nathaniel Adjei,DF,Lorient,Ligue 1,13.8,13,9,8,3,2,...,2,2,0,0,60,30,22,57.7,2023-2024,6.44


In [33]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import optuna

df = pd.read_csv("finalized_players.csv")
df = df.dropna(subset=['Rating'])

df['Defensive_Efficiency'] = (df['Tkl'] + df['Blocks'] + df['Int']) / df['90s']
df['Offensive_Contribution'] = (df['Att 3rd'] + df['Crs'] + df['Sh']) / df['90s']
df['Tactical_Contribution'] = df['TklW'] + (df['Tkl%'] * df['Tkl'])
df['Penalty_Risk'] = (df['CrdY'] + df['CrdR'] + df['PKcon']) / df['90s']
df['Defensive_Interaction'] = df['Tkl'] + df['Blocks'] + df['Int']
#df['Seasonal_Trend'] = df.groupby('Season')['Rating'].transform(lambda x: x.diff()).fillna(0)
df['Win_Ratio'] = df['Won'] / (df['Won'] + df['Lost'])

# Initialize 'Adjusted Rating' as a copy of 'Rating' to prevent NaNs for non-defenders
df['Adjusted Rating'] = df['Rating']

# Calculate mean and standard deviation for defenders only
mean_rating = df[df['Pos'] == 'DF']['Rating'].mean()
std_rating = df[df['Pos'] == 'DF']['Rating'].std()

# Calculate z-scores for defenders only
z_ratings = (df[df['Pos'] == 'DF']['Rating'] - mean_rating) / std_rating

# Increase variance by scaling z-scores (factor > 1)
scaling_factor = 1.3  # Adjust the factor based on how much you want to increase the variance
scaled_z_ratings = z_ratings * scaling_factor

# Revert to the original scale with increased variance for defenders only
df.loc[df['Pos'] == 'DF', 'Adjusted Rating'] = mean_rating + scaled_z_ratings * std_rating


#df = df[df['Pos'].str.contains("DF", na=False)]
df = df.drop(columns=['Player Name'], errors="ignore")  # Remove identifier column

df = df.drop(columns=['Squad'])

# One-hot encode categorical columns
categorical_cols = ['League', 'Season', 'Pos']
df = pd.get_dummies(df, columns=categorical_cols)

# Fill missing values
df = df.fillna(df.median())

#df = pd.get_dummies(df, columns=['Pos'], drop_first=True)

# Drop Season columns if needed
#df = df.drop(df.filter(like="Season").columns, axis=1)
#df = df.drop(df.filter(like="Squad").columns, axis=1)
#df = df.drop(df.filter(like="League").columns, axis=1)

# Split data into features (X) and target (y)
X = df.drop(columns=['Rating', 'Adjusted Rating'], errors="ignore")
y = df['Rating']
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Define Optuna objective function
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10)
    }

    # Train model
    model = xgb.XGBRegressor(**params, random_state=42)
    model.fit(X_train, y_train)

    # Validate performance
    y_pred = model.predict(X_val)
    return r2_score(y_val, y_pred)

# Run Optuna optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=1000)

# Print best parameters
best_params = study.best_params
print("Best Hyperparameters:", best_params)

# Train final model with best parameters
best_model = xgb.XGBRegressor(**best_params, random_state=42)
best_model.fit(X_train, y_train)

# Evaluate performance on test set
y_pred_test = best_model.predict(X_test)
final_r2 = r2_score(y_test, y_pred_test)
final_mae = mean_absolute_error(y_test, y_pred_test)

print(f"Final R² Score: {final_r2:.4f}")
print(f"Final MAE: {final_mae:.4f}")

[I 2025-04-10 20:03:16,886] A new study created in memory with name: no-name-67a19d71-49bf-46c6-b096-3dc1d8010c59
[I 2025-04-10 20:03:17,317] Trial 0 finished with value: 0.47249740397670925 and parameters: {'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.010238434395767002, 'subsample': 0.8634043880445559, 'colsample_bytree': 0.654719171478169, 'reg_alpha': 0.014238118656837202, 'reg_lambda': 0.004297835379600499, 'min_child_weight': 9}. Best is trial 0 with value: 0.47249740397670925.
[I 2025-04-10 20:03:18,471] Trial 1 finished with value: 0.5462435965411981 and parameters: {'n_estimators': 400, 'max_depth': 7, 'learning_rate': 0.013738675139269376, 'subsample': 0.8817880364514528, 'colsample_bytree': 0.6378747281755013, 'reg_alpha': 0.05456781352868112, 'reg_lambda': 3.2748128210834797, 'min_child_weight': 10}. Best is trial 1 with value: 0.5462435965411981.
[I 2025-04-10 20:03:18,672] Trial 2 finished with value: 0.4388641451842914 and parameters: {'n_estimators': 100, 'ma

Best Hyperparameters: {'n_estimators': 800, 'max_depth': 5, 'learning_rate': 0.014240419345231008, 'subsample': 0.5197359094137556, 'colsample_bytree': 0.7034642251910954, 'reg_alpha': 0.11692882503789641, 'reg_lambda': 0.16087361713530413, 'min_child_weight': 1}
Final R² Score: 0.5541
Final MAE: 0.1254


In [34]:
from sklearn.model_selection import cross_validate

# Define the model with best hyperparameters from Optuna
best_model = xgb.XGBRegressor(**best_params, random_state=42)

# Perform cross-validation
cv_results = cross_validate(best_model, X, y, cv=5, scoring=('r2', 'neg_mean_absolute_error'))

# Print cross-validation results
print("Cross-validation R² scores:", cv_results['test_r2'])
print("Cross-validation MAE scores:", -cv_results['test_neg_mean_absolute_error'])  # MAE is negative due to how scoring works

# Calculate the average R² score and MAE across folds
avg_r2 = cv_results['test_r2'].mean()
avg_mae = -cv_results['test_neg_mean_absolute_error'].mean()  # Negative to positive conversion for MAE

print(f"Average Cross-Validation R² Score: {avg_r2:.4f}")
print(f"Average Cross-Validation MAE: {avg_mae:.4f}")


Cross-validation R² scores: [0.49153847 0.50037792 0.48934451 0.50542692 0.56507232]
Cross-validation MAE scores: [0.13121547 0.13275953 0.12696604 0.1291835  0.12871016]
Average Cross-Validation R² Score: 0.5104
Average Cross-Validation MAE: 0.1298


In [6]:
print(list(X.columns))

['90s', 'Tkl', 'TklW', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Chl-Tkl', 'Att', 'Tkl%', 'Chl-Lost', 'Blocks', 'Sh', 'Pass', 'Int', 'Tkl+Int', 'Clr', 'Err', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Off', 'Crs', 'PKcon', 'OG', 'Recov', 'Won', 'Lost', 'Won%', 'Defensive_Efficiency', 'Offensive_Contribution', 'Tactical_Contribution', 'Penalty_Risk', 'Defensive_Interaction', 'Win_Ratio', 'Squad_Alavés', 'Squad_Almería', 'Squad_Angers', 'Squad_Arminia', 'Squad_Arsenal', 'Squad_Aston Villa', 'Squad_Atalanta', 'Squad_Athletic Club', 'Squad_Atlético Madrid', 'Squad_Augsburg', 'Squad_Auxerre', 'Squad_Barcelona', 'Squad_Bayern Munich', 'Squad_Betis', 'Squad_Bochum', 'Squad_Bologna', 'Squad_Bordeaux', 'Squad_Bournemouth', 'Squad_Brentford', 'Squad_Brest', 'Squad_Brighton', 'Squad_Burnley', 'Squad_Cagliari', 'Squad_Celta Vigo', 'Squad_Chelsea', 'Squad_Clermont Foot', 'Squad_Cremonese', 'Squad_Crystal Palace', 'Squad_Cádiz', 'Squad_Darmstadt 98', 'Squad_Dortmund', 'Squad_Eint Frankfurt', 'Squad_Elche', 'Squad_Emp

In [11]:
# Get a list of column names that contain the substring 'Squad'
squad_columns = [col for col in X.columns if 'Squad' in col]

# Print the number of columns containing 'Squad'
print(f"Number of columns containing 'Squad': {len(squad_columns)}")


Number of columns containing 'Squad': 118


In [13]:
print(list(X.columns))

['90s', 'Tkl', 'TklW', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Chl-Tkl', 'Att', 'Tkl%', 'Chl-Lost', 'Blocks', 'Sh', 'Pass', 'Int', 'Tkl+Int', 'Clr', 'Err', 'CrdY', 'CrdR', '2CrdY', 'Fls', 'Off', 'Crs', 'PKcon', 'OG', 'Recov', 'Won', 'Lost', 'Won%', 'Defensive_Efficiency', 'Offensive_Contribution', 'Tactical_Contribution', 'Penalty_Risk', 'Defensive_Interaction', 'Win_Ratio', 'Squad_Ajaccio', 'Squad_Alavés', 'Squad_Almería', 'Squad_Angers', 'Squad_Arminia', 'Squad_Arsenal', 'Squad_Aston Villa', 'Squad_Atalanta', 'Squad_Athletic Club', 'Squad_Atlético Madrid', 'Squad_Augsburg', 'Squad_Auxerre', 'Squad_Barcelona', 'Squad_Bayern Munich', 'Squad_Betis', 'Squad_Bochum', 'Squad_Bologna', 'Squad_Bordeaux', 'Squad_Bournemouth', 'Squad_Brentford', 'Squad_Brest', 'Squad_Brighton', 'Squad_Burnley', 'Squad_Cagliari', 'Squad_Celta Vigo', 'Squad_Chelsea', 'Squad_Clermont Foot', 'Squad_Cremonese', 'Squad_Crystal Palace', 'Squad_Cádiz', 'Squad_Darmstadt 98', 'Squad_Dortmund', 'Squad_Eint Frankfurt', 'Squad_E

In [22]:
print(len(df.loc[df['Pos_DF'] == True]))

1977


In [23]:
print(len(df))

3521
